# Single agent, ensemble, and hierarchy without thinking

`find_failures.ipynb` showed that with Qwen3 thinking mode off, the single agent (Qwen3-4B) gets about 47% of the development problems right, fails by wrong arithmetic rather than by running out of tokens, and uses about 2.7k tokens whatever its budget. It got about 70% at depth 2 and 25% at depth 6.

This notebook runs all three systems on the same 40 problems with thinking off for every call:

- **Single agent:** one chat with the whole budget.
- **Ensemble:** three independent chats with a third of the budget each, scored by 2-of-3 majority vote.
- **Hierarchy:** a planner, two workers, and a merge step, each released a quarter of the budget, with unused tokens carried forward.

Budgets are 12k and 16k, so each ensemble attempt (4k or 5.3k) and hierarchy stage (3k or 4k) has room for the roughly 3k tokens one solve needs. Failures here should come from reasoning, not from starved budgets.

Run it as natalia@siwek.poznan.pl, whose Drive holds the `find_failures` results, with **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/dev_systems/`, with the problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [ ]:
RUN_NAME = "dev_systems"
PROBLEMS_FROM = "find_failures/no_thinking"   # reuse that option's problems.jsonl
SYSTEMS = ["single_agent", "ensemble", "hierarchy"]
BUDGETS = [12000, 16000]
SAMPLING_SEED = 42
FLAGS = ["--no-thinking"]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/small-multi-agents/results"
OUT = f"{RESULTS}/{RUN_NAME}"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Hugging Face downloads use your token if it is stored as a Colab secret: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. The token stays in your Colab account, never in the notebook or the repo.

In [ ]:
from notebook_tools import use_hf_token

use_hf_token()

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [ ]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Problems

A copy of the `find_failures` problems, so this folder holds everything its runs used.

In [ ]:
import shutil

PROBLEMS = f"{OUT}/problems.jsonl"
shutil.copyfile(f"{RESULTS}/{PROBLEMS_FROM}/problems.jsonl", PROBLEMS)
!wc -l {PROBLEMS}

## Run

Every system at every budget, loading the model once. Everything the runner prints is also appended to `run.log`.

In [ ]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", *SYSTEMS,
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), *FLAGS, "--resume"],
           f"{OUT}/run.log")

## Results

Accuracy by system and budget. `length_limit` counts problems where the model ran out of tokens (for the ensemble, any attempt; for the hierarchy, any stage). `no_majority` is the share of ensemble problems where no two attempts agreed, and `protocol_failure` the share of hierarchy problems that stopped early for another reason: an unreadable plan or an empty report.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from notebook_tools import load_runs

runs = {path: [json.loads(line) for line in path.read_text().splitlines()]
        for path in sorted(Path(OUT).glob("b*/*.jsonl"))}
df = pd.DataFrame(load_runs(OUT))


def summarize(df, by):
    return df.groupby(by, sort=False).agg(
        accuracy=("correct", "mean"),
        problems=("correct", "size"),
        wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
        length_limit=("ran_out", "mean"),
        no_majority=("status", lambda s: (s == "no_majority").mean()),
        protocol_failure=("status", lambda s: s.isin(["invalid_plan", "empty_response"]).mean()),
        mean_total=("total", "mean"),
    ).round(2)


df.to_csv(f"{OUT}/summary_problems.csv", index=False)
by_system = summarize(df, ["system", "budget"])
by_system.to_csv(f"{OUT}/summary_by_system.csv")
by_system

Accuracy by depth and problem kind: rows are system and budget, columns depth and kind.

In [ ]:
by_depth = summarize(df, ["system", "budget", "d", "kind"])
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth["accuracy"].unstack(["d", "kind"])

## Ensemble attempts

`attempt_accuracy` is the share of single attempts that were right, `any_correct` the share of problems where at least one attempt was right, and `vote_accuracy` the share the majority vote got right. Voting can only help where attempts are right more often than not and their mistakes differ.

In [ ]:
rows = []
for path, records in runs.items():
    if path.stem != "ensemble":
        continue
    for r in records:
        statuses = [a["status"] for a in r["attempts"]]
        rows.append({"budget": r["budget"], "d": r["chain_length"], "kind": r["kind"],
                     "attempt_accuracy": statuses.count("correct") / 3,
                     "any_correct": "correct" in statuses, "vote_accuracy": r["correct"],
                     "attempt_output": sum(a["output_tokens"] for a in r["attempts"]) / 3})
if rows:
    ensemble = pd.DataFrame(rows).groupby(["budget", "d", "kind"]).mean().round(2)
    ensemble.to_csv(f"{OUT}/summary_ensemble.csv")
    display(ensemble)
else:
    print("No ensemble runs yet.")

## Hierarchy stages

How each stage spends its tokens, where unsolved problems stopped, and how often the planner asked worker 2 to use worker 1's result. Dependent problems need that hand-off; independent ones do not.

In [ ]:
import re

calls, plans = [], []
for path, records in runs.items():
    if path.stem != "hierarchy":
        continue
    for r in records:
        for call in r["calls"]:
            calls.append({"budget": r["budget"], "stage": call["stage"], "cap": call["budget"],
                          "output": call["output_tokens"], "stop_reason": call["stop_reason"]})
        found = re.search(r'"pass_first"\s*:\s*(true|false)', r["calls"][0]["final_text"]) if r["calls"] else None
        plans.append({"budget": r["budget"], "kind": r["kind"],
                      "pass_first": found.group(1) if found else "unreadable"})
if calls:
    stages = pd.DataFrame(calls).groupby(["budget", "stage"], sort=False).agg(
        calls=("output", "size"),
        mean_cap=("cap", "mean"),
        mean_output=("output", "mean"),
        max_output=("output", "max"),
        ran_out=("stop_reason", lambda s: (s != "stop").mean()),
    ).round(2)
    stages.to_csv(f"{OUT}/summary_hierarchy_stages.csv")
    display(stages)
    failed = df[(df.system == "hierarchy") & ~df.correct].groupby(
        ["budget", "failed_stage", "status"], dropna=False).size().rename("problems")
    display(failed)
    display(pd.DataFrame(plans).groupby(["budget", "kind"]).pass_first.value_counts().unstack(fill_value=0))
else:
    print("No hierarchy runs yet.")

## Compared with `find_failures`

The single agent here uses the same problems, model, sampling seed, and thinking setting as the `no_thinking` option there, so at 16k it should give mostly the same answers. vLLM batches differ between runs, so a few can change.

In [ ]:
before = pd.DataFrame(load_runs(f"{RESULTS}/{PROBLEMS_FROM}")).assign(run=PROBLEMS_FROM)
now = df[df.system == "single_agent"].assign(run=RUN_NAME)
both = pd.concat([before, now])
summarize(both, ["run", "budget"])

## Reading the results

- The hierarchy beating both the single agent and the ensemble at the same budget, most at depth 6 or on dependent problems, is the pattern the proposal predicts.
- The ensemble should help most where single attempts are right more than half the time (depth 2) and little where they are mostly wrong (depth 6).
- Many `protocol_failure` or `length_limit` results for the hierarchy point to a prompt or budget problem in one stage. The stage table shows which.
- With 10 pairs per depth and kind, each problem is 10 percentage points within a cell, so treat gaps under about 20 points as noise. The main grid needs more pairs and a test seed.